# Agentick Benchmark Dashboard
Eval **Success Rate (SR)** across PPO and LLM agents.

In [ ]:
import json
import re
import warnings
import yaml
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})

PALETTE = [
    "#E69F00", "#56B4E9", "#009E73", "#F0E442",
    "#0072B2", "#D55E00", "#CC79A7", "#000000",
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
]
DIFFICULTIES = ["easy", "medium", "hard", "expert"]
DIFF_COLORS = {"easy": "#009E73", "medium": "#0072B2", "hard": "#E69F00", "expert": "#D55E00"}
CATEGORIES = [
    "navigation", "planning", "reasoning", "memory",
    "generalization", "multi_agent",
]
TASK_CATEGORIES = {
    "GoToGoal-v0": "navigation", "MazeNavigation-v0": "navigation",
    "ShortestPath-v0": "navigation", "DynamicObstacles-v0": "navigation",
    "CuriosityMaze-v0": "navigation", "RecursiveRooms-v0": "navigation",
    "TimingChallenge-v0": "navigation", "InstructionFollowing-v0": "navigation",
    "SokobanPush-v0": "planning", "KeyDoorPuzzle-v0": "planning",
    "BacktrackPuzzle-v0": "planning", "TileSorting-v0": "planning",
    "PackingPuzzle-v0": "planning", "PreciseNavigation-v0": "planning",
    "RecipeAssembly-v0": "planning", "ToolUse-v0": "planning",
    "ResourceManagement-v0": "planning",
    "SwitchCircuit-v0": "reasoning",
    "RuleInduction-v0": "reasoning", "LightsOut-v0": "reasoning",
    "GraphColoring-v0": "reasoning", "SymbolMatching-v0": "reasoning",
    "ProgramSynthesis-v0": "reasoning", "TaskInterference-v0": "reasoning",
    "DeceptiveReward-v0": "reasoning",
    "SequenceMemory-v0": "memory", "DelayedGratification-v0": "memory",
    "TreasureHunt-v0": "memory", "FogOfWarExploration-v0": "memory",
    "FewShotAdaptation-v0": "generalization", "DistributionShift-v0": "generalization",
    "NoisyObservation-v0": "generalization",
    "CooperativeTransport-v0": "multi_agent", "TagHunt-v0": "multi_agent",
    "ChaseEvade-v0": "multi_agent", "Herding-v0": "multi_agent",
    "EmergentStrategy-v0": "multi_agent",
}

# Full expected (task, difficulty) pairs — used to fill missing results with 0
ALL_TASK_DIFFS = [(t, d) for t in TASK_CATEGORIES for d in DIFFICULTIES]
CATEGORY_TASKS = defaultdict(set)
for t, c in TASK_CATEGORIES.items():
    CATEGORY_TASKS[c].add(t)

SMOOTH_WINDOW = 5  # rolling-window size for SR curves

def smooth(y, w=SMOOTH_WINDOW):
    """Rolling mean with minimum 1-element window at edges."""
    if len(y) <= 1:
        return np.array(y)
    kernel = np.ones(w) / w
    # pad edges with first/last values
    padded = np.concatenate([[y[0]] * (w - 1), y])
    return np.convolve(padded, kernel, mode="valid")[:len(y)]

print("Imports ready.")

In [ ]:
# ── Locate results ──────────────────────────────────────────────────
def find_project_root():
    p = Path.cwd()
    for _ in range(10):
        if (p / "results").is_dir():
            return p
        p = p.parent
    raise FileNotFoundError("Cannot find results/ directory.")

ROOT = find_project_root()
RESULTS_DIR = ROOT / "results"

ALL_TASK_NAMES = set(TASK_CATEGORIES.keys())
TASK_STEMS = {t.removesuffix("-v0") for t in ALL_TASK_NAMES}

def _strip_task_suffix(name):
    for stem in sorted(TASK_STEMS, key=len, reverse=True):
        if name.endswith(f"-{stem}"):
            return name[:-(len(stem) + 1)]
    return name

def _sr_curve_from_npz(eval_dir):
    npz_path = eval_dir / "evaluations.npz"
    if not npz_path.exists():
        return []
    data = np.load(str(npz_path))
    return [{"timestep": int(ts), "success_rate": float(np.mean(data["results"][i] > 0))}
            for i, ts in enumerate(data["timesteps"])]


# ── Load PPO runs ──────────────────────────────────────────────────
# Discover runs by iterating directories (not just training_summary.json)
# so that runs without a summary (e.g. iso) are also found.
ppo_records, ppo_runs = [], {}

ppo_base = RESULTS_DIR / "ppo_benchmarks"
if ppo_base.exists():
    for run_path in sorted(ppo_base.iterdir()):
        if not run_path.is_dir():
            continue
        per_task_dir = run_path / "per_task"
        if not per_task_dir.exists():
            continue

        # Determine reward_mode + render_mode from training_summary or config.yaml
        reward_mode, render_mode = "unknown", "rgb_array"
        ts_file = run_path / "training_summary.json"
        cfg_file = run_path / "config.yaml"
        if ts_file.exists():
            with open(ts_file) as f:
                ts_data = json.load(f)
            reward_mode = ts_data.get("reward_mode", reward_mode)
            render_mode = ts_data.get("render_mode", render_mode)
        elif cfg_file.exists():
            with open(cfg_file) as f:
                cfg = yaml.safe_load(f)
            reward_mode = cfg.get("reward_mode", reward_mode)
            rms = cfg.get("render_modes", [render_mode])
            render_mode = rms[0] if rms else render_mode

        render_short = "flat" if "flat" in render_mode else "iso"
        agent_label = f"PPO ({reward_mode}, {render_short})"

        if agent_label not in ppo_runs:
            ppo_runs[agent_label] = {
                "path": run_path, "agent_label": agent_label, "agent_type": "ppo",
                "reward_mode": reward_mode, "render_mode": render_mode,
                "run_dirs": [],
            }
        ppo_runs[agent_label]["run_dirs"].append(run_path)

        # Load per_task metrics
        for task_dir in sorted(per_task_dir.iterdir()):
            if not task_dir.is_dir():
                continue
            task_name = task_dir.name
            for diff_dir in sorted(task_dir.iterdir()):
                if not diff_dir.is_dir():
                    continue
                mf = diff_dir / "metrics.json"
                if not mf.exists():
                    continue
                with open(mf) as f:
                    m = json.load(f)
                task = m.get("task", task_name)
                diff = m.get("difficulty", diff_dir.name)
                cat = m.get("category", TASK_CATEGORIES.get(task, "unknown"))
                sr_curve = []
                tc = m.get("training_curve", [])
                if tc and isinstance(tc[0], dict) and "success_rate" in tc[0]:
                    sr_curve = [{"timestep": p["timestep"], "success_rate": p["success_rate"]}
                                for p in tc]
                else:
                    sr_curve = _sr_curve_from_npz(diff_dir / "eval")
                ppo_records.append({
                    "run": agent_label, "agent_label": agent_label, "agent_type": "ppo",
                    "task": task, "task_short": task.replace("-v0", ""),
                    "difficulty": diff, "category": cat,
                    "success_rate": m.get("success_rate", 0.0),
                    "mean_return": m.get("mean_return", 0.0),
                    "sr_curve": sr_curve,
                })

# Deduplicate
seen = set()
ppo_records = [r for r in ppo_records
               if (k := (r["run"], r["task"], r["difficulty"])) not in seen and not seen.add(k)]

print(f"PPO: {len(ppo_runs)} agent(s), {len(ppo_records)} records")


# ── Load LLM/API runs ─────────────────────────────────────────────
llm_records, llm_runs = [], {}

llm_base = RESULTS_DIR / "llm_benchmarks"
if llm_base.exists():
    for summary_file in sorted(llm_base.rglob("summary.json")):
        run_path = summary_file.parent
        cfg_path = run_path / "config.yaml"
        agent_label = run_path.name
        if cfg_path.exists():
            m_name = re.search(r"^name:\s*(.+)$", cfg_path.read_text(), re.MULTILINE)
            if m_name:
                agent_label = _strip_task_suffix(m_name.group(1).strip())
        if agent_label not in llm_runs:
            llm_runs[agent_label] = {
                "path": run_path, "agent_label": agent_label, "agent_type": "llm",
                "run_dirs": [],
            }
        llm_runs[agent_label]["run_dirs"].append(run_path)
        per_task_dir = run_path / "per_task"
        if not per_task_dir.exists():
            continue
        for task_dir in sorted(per_task_dir.iterdir()):
            if not task_dir.is_dir():
                continue
            mf = task_dir / "metrics.json"
            if not mf.exists():
                continue
            with open(mf) as f:
                td = json.load(f)
            task_name = td.get("task_name", task_dir.name)
            cat = TASK_CATEGORIES.get(task_name, "unknown")
            for diff, dd in td.get("per_difficulty", {}).items():
                metrics = dd.get("metrics", {})
                llm_records.append({
                    "run": agent_label, "agent_label": agent_label, "agent_type": "llm",
                    "task": task_name, "task_short": task_name.replace("-v0", ""),
                    "difficulty": diff, "category": cat,
                    "success_rate": metrics.get("success_rate", 0.0),
                    "mean_return": metrics.get("mean_return", 0.0),
                    "sr_curve": [],
                })

seen2 = set()
llm_records = [r for r in llm_records
               if (k := (r["run"], r["task"], r["difficulty"])) not in seen2 and not seen2.add(k)]
print(f"LLM: {len(llm_runs)} agent(s), {len(llm_records)} records")


# ── Load baseline runs (oracle, random) ────────────────────────────
baseline_records, baseline_runs = [], {}

baseline_base = RESULTS_DIR / "baseline_benchmarks"
if baseline_base.exists():
    for summary_file in sorted(baseline_base.rglob("summary.json")):
        run_path = summary_file.parent
        cfg_path = run_path / "config.yaml"
        agent_label = run_path.name
        if cfg_path.exists():
            m_name = re.search(r"^name:\s*(.+)$", cfg_path.read_text(), re.MULTILINE)
            if m_name:
                agent_label = _strip_task_suffix(m_name.group(1).strip())
        if agent_label not in baseline_runs:
            baseline_runs[agent_label] = {
                "path": run_path, "agent_label": agent_label, "agent_type": "baseline",
                "run_dirs": [],
            }
        baseline_runs[agent_label]["run_dirs"].append(run_path)
        per_task_dir = run_path / "per_task"
        if not per_task_dir.exists():
            continue
        for task_dir in sorted(per_task_dir.iterdir()):
            if not task_dir.is_dir():
                continue
            mf = task_dir / "metrics.json"
            if not mf.exists():
                continue
            with open(mf) as f:
                td = json.load(f)
            task_name = td.get("task_name", task_dir.name)
            cat = TASK_CATEGORIES.get(task_name, "unknown")
            for diff, dd in td.get("per_difficulty", {}).items():
                metrics = dd.get("metrics", {})
                baseline_records.append({
                    "run": agent_label, "agent_label": agent_label, "agent_type": "baseline",
                    "task": task_name, "task_short": task_name.replace("-v0", ""),
                    "difficulty": diff, "category": cat,
                    "success_rate": metrics.get("success_rate", 0.0),
                    "mean_return": metrics.get("mean_return", 0.0),
                    "sr_curve": [],
                })

seen3 = set()
baseline_records = [r for r in baseline_records
               if (k := (r["run"], r["task"], r["difficulty"])) not in seen3 and not seen3.add(k)]
print(f"Baseline: {len(baseline_runs)} agent(s), {len(baseline_records)} records")


# ── Load VLM runs ─────────────────────────────────────────────────
vlm_records, vlm_runs = [], {}

vlm_base = RESULTS_DIR / "vlm_benchmarks"
if vlm_base.exists():
    for summary_file in sorted(vlm_base.rglob("summary.json")):
        run_path = summary_file.parent
        cfg_path = run_path / "config.yaml"
        agent_label = run_path.name
        if cfg_path.exists():
            m_name = re.search(r"^name:\s*(.+)$", cfg_path.read_text(), re.MULTILINE)
            if m_name:
                agent_label = _strip_task_suffix(m_name.group(1).strip())
        if agent_label not in vlm_runs:
            vlm_runs[agent_label] = {
                "path": run_path, "agent_label": agent_label, "agent_type": "vlm",
                "run_dirs": [],
            }
        vlm_runs[agent_label]["run_dirs"].append(run_path)
        per_task_dir = run_path / "per_task"
        if not per_task_dir.exists():
            continue
        for task_dir in sorted(per_task_dir.iterdir()):
            if not task_dir.is_dir():
                continue
            mf = task_dir / "metrics.json"
            if not mf.exists():
                continue
            with open(mf) as f:
                td = json.load(f)
            task_name = td.get("task_name", task_dir.name)
            cat = TASK_CATEGORIES.get(task_name, "unknown")
            for diff, dd in td.get("per_difficulty", {}).items():
                metrics = dd.get("metrics", {})
                vlm_records.append({
                    "run": agent_label, "agent_label": agent_label, "agent_type": "vlm",
                    "task": task_name, "task_short": task_name.replace("-v0", ""),
                    "difficulty": diff, "category": cat,
                    "success_rate": metrics.get("success_rate", 0.0),
                    "mean_return": metrics.get("mean_return", 0.0),
                    "sr_curve": [],
                })

seen4 = set()
vlm_records = [r for r in vlm_records
               if (k := (r["run"], r["task"], r["difficulty"])) not in seen4 and not seen4.add(k)]
print(f"VLM: {len(vlm_runs)} agent(s), {len(vlm_records)} records")


# ── Load API benchmark runs ───────────────────────────────────────
# API benchmarks store results as trace files per difficulty:
#   per_task/<Task>/traces/<difficulty>/<seed>.json
# Each trace has metadata with 'success', 'total_reward', 'difficulty'.
# summary.json has overall cost info.
api_records, api_runs = [], {}

api_base = RESULTS_DIR / "api_benchmarks"
if api_base.exists():
    for run_path in sorted(api_base.iterdir()):
        if not run_path.is_dir():
            continue
        summary_file = run_path / "summary.json"
        if not summary_file.exists():
            continue
        cfg_path = run_path / "config.yaml"
        agent_label = run_path.name
        if cfg_path.exists():
            m_name = re.search(r"^name:\s*(.+)$", cfg_path.read_text(), re.MULTILINE)
            if m_name:
                agent_label = _strip_task_suffix(m_name.group(1).strip())

        # Load cost info from summary
        with open(summary_file) as f:
            summary = json.load(f)
        cost_usd = summary.get("total_cost_usd", 0.0)
        cost_report = summary.get("cost_report", {})

        if agent_label not in api_runs:
            api_runs[agent_label] = {
                "path": run_path, "agent_label": agent_label, "agent_type": "api",
                "run_dirs": [], "total_cost_usd": 0.0, "cost_report": {},
            }
        api_runs[agent_label]["run_dirs"].append(run_path)
        api_runs[agent_label]["total_cost_usd"] += cost_usd
        api_runs[agent_label]["cost_report"] = cost_report

        per_task_dir = run_path / "per_task"
        if not per_task_dir.exists():
            continue
        for task_dir in sorted(per_task_dir.iterdir()):
            if not task_dir.is_dir():
                continue
            task_name = task_dir.name
            cat = TASK_CATEGORIES.get(task_name, "unknown")
            traces_dir = task_dir / "traces"
            if not traces_dir.exists():
                continue
            for diff_dir in sorted(traces_dir.iterdir()):
                if not diff_dir.is_dir():
                    continue
                diff = diff_dir.name
                successes = []
                rewards = []
                for trace_file in sorted(diff_dir.iterdir()):
                    if not trace_file.suffix == ".json":
                        continue
                    with open(trace_file) as f:
                        trace = json.load(f)
                    meta = trace.get("metadata", {})
                    successes.append(1.0 if meta.get("success", False) else 0.0)
                    rewards.append(meta.get("total_reward", 0.0))
                if successes:
                    api_records.append({
                        "run": agent_label, "agent_label": agent_label, "agent_type": "api",
                        "task": task_name, "task_short": task_name.replace("-v0", ""),
                        "difficulty": diff, "category": cat,
                        "success_rate": float(np.mean(successes)),
                        "mean_return": float(np.mean(rewards)),
                        "sr_curve": [],
                    })

seen5 = set()
api_records = [r for r in api_records
               if (k := (r["run"], r["task"], r["difficulty"])) not in seen5 and not seen5.add(k)]
print(f"API: {len(api_runs)} agent(s), {len(api_records)} records")
for rn, info in api_runs.items():
    print(f"  {rn}: ${info['total_cost_usd']:.2f}")


# ── Combine + fill missing (task, diff) with SR=0 ─────────────────
# If an agent crashed on some tasks, those missing pairs count as 0 SR
# so category averages aren't inflated by only counting successes.
all_records = ppo_records + llm_records + vlm_records + baseline_records + api_records
all_runs = {**ppo_runs, **llm_runs, **vlm_runs, **baseline_runs, **api_runs}
run_names = sorted(all_runs.keys())

for rn in run_names:
    existing = {(r["task"], r["difficulty"]) for r in all_records if r["run"] == rn}
    for task, diff in ALL_TASK_DIFFS:
        if (task, diff) not in existing:
            all_records.append({
                "run": rn, "agent_label": all_runs[rn]["agent_label"],
                "agent_type": all_runs[rn]["agent_type"],
                "task": task, "task_short": task.replace("-v0", ""),
                "difficulty": diff, "category": TASK_CATEGORIES[task],
                "success_rate": 0.0, "mean_return": 0.0, "sr_curve": [],
            })

agent_colors = {rn: PALETTE[i % len(PALETTE)] for i, rn in enumerate(run_names)}

print(f"\nTotal: {len(all_runs)} agent(s), {len(all_records)} records")
for rn in run_names:
    n = sum(1 for r in all_records if r["run"] == rn)
    real = sum(1 for r in all_records if r["run"] == rn and r["success_rate"] > 0)
    print(f"  {all_runs[rn]['agent_label']:40s} [{all_runs[rn]['agent_type']:>8s}] {n:>4d} records ({real} nonzero)")

In [ ]:
# ── Overall SR barplot + Category grid ─────────────────────────────
if not run_names:
    print("No results found.")
else:
    # Sort agents by overall SR
    agent_sr = {}
    for rn in run_names:
        srs = [r["success_rate"] for r in all_records if r["run"] == rn]
        agent_sr[rn] = np.mean(srs) if srs else 0
    sorted_agents = sorted(agent_sr, key=lambda rn: agent_sr[rn], reverse=True)
    labels = [all_runs[rn]["agent_label"] for rn in sorted_agents]
    colors = [agent_colors[rn] for rn in sorted_agents]

    # ---- Overall barplot ----
    fig, ax = plt.subplots(figsize=(max(6, len(sorted_agents) * 1.2), 4))
    x = np.arange(len(sorted_agents))
    means = [agent_sr[rn] for rn in sorted_agents]
    ax.bar(x, means, color=colors, edgecolor="white", linewidth=0.5)
    for i, m in enumerate(means):
        ax.text(i, m + 0.02, f"{m:.1%}", ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Mean Eval SR")
    ax.set_ylim(0, 1.05)
    ax.set_title("Overall Benchmark: Eval Success Rate", fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ---- Category grid (2x3) ----
    active_cats = [c for c in CATEGORIES if any(r["category"] == c for r in all_records)]
    n_cols, n_rows = 3, 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 7), squeeze=False)

    for ci, cat in enumerate(active_cats):
        ax = axes[ci // n_cols][ci % n_cols]
        cat_means = []
        for rn in sorted_agents:
            srs = [r["success_rate"] for r in all_records
                   if r["run"] == rn and r["category"] == cat]
            cat_means.append(np.mean(srs) if srs else 0)
        bars = ax.bar(np.arange(len(sorted_agents)), cat_means, color=colors,
                      edgecolor="white", linewidth=0.5)
        ax.set_title(cat.replace("_", " ").title(), fontweight="bold", fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.set_xticks(np.arange(len(sorted_agents)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=6)
        for i, m in enumerate(cat_means):
            if m > 0.01:
                ax.text(i, m + 0.02, f"{m:.0%}", ha="center", va="bottom", fontsize=6)

    # Hide unused subplots
    for ci in range(len(active_cats), n_rows * n_cols):
        axes[ci // n_cols][ci % n_cols].set_visible(False)

    fig.suptitle("Eval SR by Category", fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── PPO Eval SR Training Curves (by category, smoothed) ───────────
ppo_run_names = [rn for rn in run_names if all_runs[rn]["agent_type"] == "ppo"]

for rn in ppo_run_names:
    recs_with_curves = [r for r in ppo_records if r["run"] == rn and r["sr_curve"]]
    if not recs_with_curves:
        print(f"No SR curves for {all_runs[rn]['agent_label']}")
        continue

    cat_recs = defaultdict(list)
    for r in recs_with_curves:
        cat_recs[r["category"]].append(r)
    active_cats = [c for c in CATEGORIES if c in cat_recs]
    if not active_cats:
        continue

    n_cols, n_rows = 3, (len(active_cats) + 2) // 3
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False)

    for idx, cat in enumerate(active_cats):
        ax = axes[idx // n_cols][idx % n_cols]
        for ti, r in enumerate(sorted(cat_recs[cat], key=lambda x: x["task"])):
            curve = r["sr_curve"]
            ts = np.array([0] + [p["timestep"] for p in curve])
            srs = np.array([0.0] + [p["success_rate"] for p in curve])
            srs_smooth = smooth(srs)
            color = PALETTE[ti % len(PALETTE)]
            ax.plot(ts, srs_smooth, label=r["task_short"], color=color, linewidth=1.2)

        ax.set_title(cat.replace("_", " ").title(), fontsize=10, fontweight="bold")
        ax.set_xlabel("Timesteps")
        ax.set_ylabel("Eval SR")
        ax.set_xlim(left=0)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=5, loc="best", ncol=2)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

    for idx in range(len(active_cats), n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    fig.suptitle(f"PPO Eval SR During Training — {all_runs[rn]['agent_label']}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Normalized Performance (HNS-style) ────────────────────────────
# HNS = (agent_SR - random_SR) / (oracle_SR - random_SR)
# Clipped to [0, 1] range

# Build lookup tables for baselines
random_sr = {}  # (task, difficulty) -> SR
oracle_sr = {}  # (task, difficulty) -> SR

for r in all_records:
    if r["agent_label"] == "random_agent":
        random_sr[(r["task"], r["difficulty"])] = r["success_rate"]
    elif r["agent_label"] == "oracle_agent":
        oracle_sr[(r["task"], r["difficulty"])] = r["success_rate"]

if not random_sr and not oracle_sr:
    print("No oracle/random baseline results found. Run oracle_agent and random_agent benchmarks first.")
    print("  python examples/experiments/slurm/launch.py --configs oracle_agent random_agent")
else:
    def compute_hns(sr, task, diff):
        """Compute Human-Normalized Score."""
        r = random_sr.get((task, diff), 0.0)
        o = oracle_sr.get((task, diff), 1.0)
        if o - r < 1e-6:
            return 1.0 if sr >= o else 0.0
        return np.clip((sr - r) / (o - r), 0.0, 1.0)

    # Compute HNS for all non-baseline agents
    hns_agents = [rn for rn in run_names if rn not in ("random_agent", "oracle_agent")]

    if hns_agents:
        # Overall HNS barplot
        agent_hns = {}
        for rn in hns_agents:
            hns_vals = []
            for r in all_records:
                if r["run"] == rn:
                    hns_vals.append(compute_hns(r["success_rate"], r["task"], r["difficulty"]))
            agent_hns[rn] = np.mean(hns_vals) if hns_vals else 0

        sorted_hns = sorted(agent_hns, key=lambda rn: agent_hns[rn], reverse=True)
        labels = [all_runs[rn]["agent_label"] for rn in sorted_hns]
        colors = [agent_colors.get(rn, "#999999") for rn in sorted_hns]

        fig, ax = plt.subplots(figsize=(max(6, len(sorted_hns) * 1.2), 4))
        x = np.arange(len(sorted_hns))
        means = [agent_hns[rn] for rn in sorted_hns]
        ax.bar(x, means, color=colors, edgecolor="white", linewidth=0.5)
        for i, m in enumerate(means):
            ax.text(i, m + 0.02, f"{m:.1%}", ha="center", va="bottom", fontsize=8, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Mean HNS")
        ax.set_ylim(0, 1.05)
        ax.set_title("Normalized Performance (HNS): (Agent - Random) / (Oracle - Random)",
                      fontweight="bold", fontsize=11)
        plt.tight_layout()
        plt.show()

        # Category grid HNS (2x3)
        active_cats = [c for c in CATEGORIES if any(r["category"] == c for r in all_records)]
        n_cols, n_rows = 3, 2
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 7), squeeze=False)

        for ci, cat in enumerate(active_cats):
            ax = axes[ci // n_cols][ci % n_cols]
            cat_hns = []
            for rn in sorted_hns:
                hns_vals = [compute_hns(r["success_rate"], r["task"], r["difficulty"])
                            for r in all_records
                            if r["run"] == rn and r["category"] == cat]
                cat_hns.append(np.mean(hns_vals) if hns_vals else 0)
            ax.bar(np.arange(len(sorted_hns)), cat_hns, color=colors,
                   edgecolor="white", linewidth=0.5)
            ax.set_title(cat.replace("_", " ").title(), fontweight="bold", fontsize=10)
            ax.set_ylim(0, 1.05)
            ax.set_xticks(np.arange(len(sorted_hns)))
            ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=6)
            for i, m in enumerate(cat_hns):
                if m > 0.01:
                    ax.text(i, m + 0.02, f"{m:.0%}", ha="center", va="bottom", fontsize=6)

        for ci in range(len(active_cats), n_rows * n_cols):
            axes[ci // n_cols][ci % n_cols].set_visible(False)

        fig.suptitle("Normalized Performance (HNS) by Category", fontweight="bold", fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print("No non-baseline agents found for HNS computation.")